# AI Агент с доступом к инструментам и метриками оценки

Этот ноутбук демонстрирует AI агента, который может:
1. Получать доступ к внешним инструментам (например, API маркетплейса Ozon)
2. Принимать интеллектуальные решения об использовании этих инструментов
3. Оценивать свою собственную производительность с помощью комплексных метрик

In [1]:
import json
import time
from typing import Any, Dict, List, Optional, Callable
from dataclasses import dataclass, field
from enum import Enum
import statistics
from datetime import datetime

# Для демонстрации мы будем использовать имитацию LLM в стиле OpenAI
try:
    from openai import OpenAI
    HAS_OPENAI = True
except ImportError:
    HAS_OPENAI = False
    print("OpenAI не установлен. Используем имитационные ответы.")

OpenAI не установлен. Используем имитационные ответы.


In [2]:
# ============================================================================
# ИМИТАЦИЯ API МАРКЕТПЛЕЙСА OZON
# ============================================================================

# Имитированная база данных маркетплейса
MARKETPLACE_DATA = {
    "products": {
        "laptop": [
            {"id": 1, "name": "Dell XPS 13", "price": 89999, "rating": 4.8, "stock": 15},
            {"id": 2, "name": "MacBook Air M3", "price": 119999, "rating": 4.9, "stock": 8},
            {"id": 3, "name": "ASUS VivoBook", "price": 54999, "rating": 4.5, "stock": 25},
            {"id": 4, "name": "HP Pavilion", "price": 59999, "rating": 4.3, "stock": 20},
        ],
        "smartphone": [
            {"id": 5, "name": "iPhone 15 Pro", "price": 99999, "rating": 4.9, "stock": 5},
            {"id": 6, "name": "Samsung Galaxy S24", "price": 79999, "rating": 4.7, "stock": 12},
            {"id": 7, "name": "Google Pixel 8", "price": 69999, "rating": 4.6, "stock": 18},
            {"id": 8, "name": "Xiaomi 14", "price": 49999, "rating": 4.4, "stock": 30},
        ],
        "headphones": [
            {"id": 9, "name": "Sony WH-1000XM5", "price": 29999, "rating": 4.8, "stock": 40},
            {"id": 10, "name": "Apple AirPods Pro", "price": 19999, "rating": 4.7, "stock": 50},
            {"id": 11, "name": "Bose NC 700", "price": 34999, "rating": 4.6, "stock": 25},
            {"id": 12, "name": "JBL Tune 770NC", "price": 12999, "rating": 4.2, "stock": 60},
        ],
    }
}


def search_marketplace(product_category: str, max_results: int = 5) -> Dict[str, Any]:
    """Поиск продуктов в маркетплейсе по категории."""
    if product_category not in MARKETPLACE_DATA["products"]:
        return {"status": "error", "message": f"Категория '{product_category}' не найдена"}
    
    products = MARKETPLACE_DATA["products"][product_category][:max_results]
    return {
        "status": "success",
        "category": product_category,
        "products": products,
        "total_count": len(MARKETPLACE_DATA["products"][product_category])
    }


def get_product_details(product_id: int) -> Dict[str, Any]:
    """Получить подробную информацию о конкретном продукте."""
    for category, products in MARKETPLACE_DATA["products"].items():
        for product in products:
            if product["id"] == product_id:
                return {
                    "status": "success",
                    "product": product,
                    "category": category,
                    "availability": "in_stock" if product["stock"] > 0 else "out_of_stock"
                }
    return {"status": "error", "message": f"Продукт с id {product_id} не найден"}


def compare_products(product_ids: List[int]) -> Dict[str, Any]:
    """Сравнить несколько продуктов по их спецификациям."""
    products = []
    for product_id in product_ids:
        result = get_product_details(product_id)
        if result["status"] == "success":
            products.append(result["product"])
    
    if not products:
        return {"status": "error", "message": "Нет допустимых продуктов для сравнения"}
    
    avg_price = statistics.mean([p["price"] for p in products])
    avg_rating = statistics.mean([p["rating"] for p in products])
    total_stock = sum([p["stock"] for p in products])
    
    return {
        "status": "success",
        "compared_products": products,
        "comparison": {
            "price_range": (min([p["price"] for p in products]), max([p["price"] for p in products])),
            "average_price": round(avg_price, 2),
            "average_rating": round(avg_rating, 2),
            "total_stock": total_stock,
            "best_value": min(products, key=lambda p: p["price"])["name"],
            "best_rated": max(products, key=lambda p: p["rating"])["name"],
        }
    }

print("✓ Инициализировано имитационное API маркетплейса Ozon")

✓ Инициализировано имитационное API маркетплейса Ozon


In [3]:
# ============================================================================
# ОПРЕДЕЛЕНИЯ ИНСТРУМЕНТОВ
# ============================================================================

@dataclass
class Tool:
    """Определение инструмента, который может использовать агент."""
    name: str
    description: str
    parameters: Dict[str, Any]
    function: Callable
    
    def to_dict(self) -> Dict[str, Any]:
        """Преобразовать инструмент в формат словаря для LLM."""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": self.parameters["properties"],
                    "required": self.parameters.get("required", [])
                }
            }
        }


# Определить доступные инструменты
TOOLS = [
    Tool(
        name="search_marketplace",
        description="Поиск продуктов в маркетплейсе по категории",
        parameters={
            "properties": {
                "product_category": {
                    "type": "string",
                    "description": "Категория для поиска (laptop, smartphone, headphones)"
                },
                "max_results": {
                    "type": "integer",
                    "description": "Максимальное количество результатов для возврата"
                }
            },
            "required": ["product_category"]
        },
        function=search_marketplace
    ),
    Tool(
        name="get_product_details",
        description="Получить подробную информацию о конкретном продукте",
        parameters={
            "properties": {
                "product_id": {
                    "type": "integer",
                    "description": "ID продукта для получения деталей"
                }
            },
            "required": ["product_id"]
        },
        function=get_product_details
    ),
    Tool(
        name="compare_products",
        description="Сравнить несколько продуктов по спецификациям и цене",
        parameters={
            "properties": {
                "product_ids": {
                    "type": "array",
                    "items": {"type": "integer"},
                    "description": "Список ID продуктов для сравнения"
                }
            },
            "required": ["product_ids"]
        },
        function=compare_products
    )
]

print(f"✓ Зарегистрировано {len(TOOLS)} инструментов: {[t.name for t in TOOLS]}")

✓ Зарегистрировано 3 инструментов: ['search_marketplace', 'get_product_details', 'compare_products']


In [4]:
# ============================================================================
# ЯДРО AI АГЕНТА
# ============================================================================

@dataclass
class AgentAction:
    """Представляет действие, выполненное агентом."""
    tool_name: str
    tool_input: Dict[str, Any]
    timestamp: float = field(default_factory=time.time)
    execution_time: Optional[float] = None
    result: Optional[Dict[str, Any]] = None
    success: bool = False


@dataclass
class AgentTrace:
    """Полная трассировка выполнения агента."""
    user_query: str
    actions: List[AgentAction] = field(default_factory=list)
    final_response: str = ""
    total_execution_time: float = 0.0
    started_at: datetime = field(default_factory=datetime.now)
    completed_at: Optional[datetime] = None
    
    def to_dict(self) -> Dict[str, Any]:
        """Преобразовать трассировку в словарь."""
        return {
            "user_query": self.user_query,
            "num_actions": len(self.actions),
            "actions": [
                {
                    "tool": action.tool_name,
                    "input": action.tool_input,
                    "success": action.success,
                    "execution_time": action.execution_time
                }
                for action in self.actions
            ],
            "final_response": self.final_response,
            "total_execution_time": self.total_execution_time,
            "duration": (self.completed_at - self.started_at).total_seconds() if self.completed_at else 0
        }


class SimpleAgent:
    """Простой AI агент, который может использовать инструменты для ответа на вопросы."""
    
    def __init__(self, tools: List[Tool], use_llm: bool = False):
        self.tools = {tool.name: tool for tool in tools}
        self.use_llm = use_llm and HAS_OPENAI
        self.execution_traces: List[AgentTrace] = []
        
    def _mock_agent_response(self, query: str) -> tuple[str, List[str]]:
        """Генерировать имитационный ответ без вызова LLM."""
        query_lower = query.lower()
        
        # Простые эвристики для выбора инструмента
        if "compare" in query_lower:
            tools = ["compare_products"]
        elif "detail" in query_lower or "information" in query_lower or "info" in query_lower:
            tools = ["get_product_details"]
        elif "search" in query_lower or "find" in query_lower:
            tools = ["search_marketplace"]
        else:
            tools = ["search_marketplace"]  # по умолчанию
            
        return f"Я помогу вам найти лучшие варианты. Давайте поищем на маркетплейсе.", tools
    
    def _execute_tool(self, tool_name: str, tool_input: Dict[str, Any]) -> Dict[str, Any]:
        """Выполнить инструмент и вернуть результат."""
        if tool_name not in self.tools:
            return {"error": f"Инструмент '{tool_name}' не найден"}
        
        tool = self.tools[tool_name]
        try:
            result = tool.function(**tool_input)
            return result
        except Exception as e:
            return {"error": f"Выполнение инструмента не удалось: {str(e)}"}
    
    def process_query(self, query: str, max_iterations: int = 5) -> str:
        """Обработать запрос пользователя и вернуть ответ."""
        trace = AgentTrace(user_query=query)
        
        # Начальное решение агента
        initial_response, tool_names = self._mock_agent_response(query)
        
        # Выполнить инструменты
        tool_results = []
        for tool_name in tool_names[:max_iterations]:
            if tool_name not in self.tools:
                continue
                
            # Для этой простой демонстрации мы будем решать входные данные на основе распространенных запросов
            if tool_name == "search_marketplace":
                tool_input = {"product_category": "laptop"}
                if "smartphone" in query.lower():
                    tool_input["product_category"] = "smartphone"
                elif "headphone" in query.lower():
                    tool_input["product_category"] = "headphones"
                if "all" in query.lower():
                    tool_input["max_results"] = 10
            elif tool_name == "get_product_details":
                tool_input = {"product_id": 1}
            elif tool_name == "compare_products":
                tool_input = {"product_ids": [1, 2, 3]}
            else:
                tool_input = {}
            
            # Записать действие
            action = AgentAction(
                tool_name=tool_name,
                tool_input=tool_input,
                timestamp=time.time()
            )
            
            # Выполнить
            start_time = time.time()
            result = self._execute_tool(tool_name, tool_input)
            action.execution_time = time.time() - start_time
            action.result = result
            action.success = result.get("status") == "success"
            
            trace.actions.append(action)
            tool_results.append(result)
        
        # Генерировать окончательный ответ
        response = f"{initial_response}\n\n"
        for i, result in enumerate(tool_results):
            if result.get("status") == "success":
                if "products" in result:
                    response += f"Найдено {len(result['products'])} продуктов:\n"
                    for p in result['products']:
                        response += f"  • {p['name']}: ₽{p['price']:,} (Рейтинг: {p['rating']}/5)\n"
                if "comparison" in result:
                    comp = result["comparison"]
                    response += f"\nРезультаты сравнения:\n"
                    response += f"  Диапазон цен: ₽{comp['price_range'][0]:,} - ₽{comp['price_range'][1]:,}\n"
                    response += f"  Средняя цена: ₽{comp['average_price']:,}\n"
                    response += f"  Средний рейтинг: {comp['average_rating']}/5\n"
                    response += f"  Лучшая цена: {comp['best_value']}\n"
                    response += f"  Лучший рейтинг: {comp['best_rated']}\n"
        
        trace.final_response = response
        trace.completed_at = datetime.now()
        trace.total_execution_time = sum([a.execution_time or 0 for a in trace.actions])
        
        self.execution_traces.append(trace)
        return response

print("✓ Класс SimpleAgent инициализирован")

✓ Класс SimpleAgent инициализирован


In [5]:
# ============================================================================
# МЕТРИКИ ОЦЕНКИ
# ============================================================================

class MetricType(Enum):
    """Типы метрик для отслеживания."""
    EXECUTION_TIME = "execution_time"
    TOOL_USAGE = "tool_usage"
    SUCCESS_RATE = "success_rate"
    QUERY_QUALITY = "query_quality"


@dataclass
class Metrics:
    """Метрики производительности агента."""
    total_queries: int = 0
    successful_queries: int = 0
    total_tool_calls: int = 0
    successful_tool_calls: int = 0
    total_execution_time: float = 0.0
    average_execution_time: float = 0.0
    tool_usage_count: Dict[str, int] = field(default_factory=dict)
    tool_success_rate: Dict[str, float] = field(default_factory=dict)
    query_complexity_distribution: Dict[str, int] = field(default_factory=lambda: {"simple": 0, "medium": 0, "complex": 0})
    
    def to_dict(self) -> Dict[str, Any]:
        """Преобразовать метрики в словарь."""
        return {
            "total_queries": self.total_queries,
            "successful_queries": self.successful_queries,
            "success_rate": f"{(self.successful_queries / self.total_queries * 100):.2f}%" if self.total_queries > 0 else "0%",
            "total_tool_calls": self.total_tool_calls,
            "successful_tool_calls": self.successful_tool_calls,
            "tool_success_rate": f"{(self.successful_tool_calls / self.total_tool_calls * 100):.2f}%" if self.total_tool_calls > 0 else "0%",
            "average_execution_time": f"{self.average_execution_time:.4f}s",
            "tool_usage_distribution": self.tool_usage_count,
            "tool_success_rates": self.tool_success_rate,
            "query_complexity": self.query_complexity_distribution
        }


class MetricsCollector:
    """Собирает и агрегирует метрики из выполнения агента."""
    
    def __init__(self):
        self.metrics = Metrics()
        self.execution_times: List[float] = []
        self.tool_results: Dict[str, List[bool]] = {}
        
    def record_trace(self, trace: AgentTrace) -> None:
        """Записать метрики из трассировки выполнения."""
        # Подсчитать запрос
        self.metrics.total_queries += 1
        
        # Определить, успешный ли
        all_successful = all(action.success for action in trace.actions)
        if all_successful and trace.actions:
            self.metrics.successful_queries += 1
        
        # Обработать действия
        for action in trace.actions:
            self.metrics.total_tool_calls += 1
            if action.success:
                self.metrics.successful_tool_calls += 1
            
            # Отслеживать использование инструмента
            if action.tool_name not in self.metrics.tool_usage_count:
                self.metrics.tool_usage_count[action.tool_name] = 0
                self.tool_results[action.tool_name] = []
            
            self.metrics.tool_usage_count[action.tool_name] += 1
            self.tool_results[action.tool_name].append(action.success)
            
            # Отслеживать время выполнения
            if action.execution_time:
                self.metrics.total_execution_time += action.execution_time
                self.execution_times.append(action.execution_time)
        
        # Рассчитать сложность
        num_actions = len(trace.actions)
        query_len = len(trace.user_query.split())
        complexity = "simple" if num_actions <= 1 else ("complex" if num_actions > 2 else "medium")
        self.metrics.query_complexity_distribution[complexity] += 1
        
        # Обновить средние
        if self.execution_times:
            self.metrics.average_execution_time = statistics.mean(self.execution_times)
        
        # Обновить показатели успешности инструментов
        for tool_name, results in self.tool_results.items():
            if results:
                success_rate = sum(results) / len(results) * 100
                self.metrics.tool_success_rate[tool_name] = f"{success_rate:.2f}%"
    
    def get_metrics(self) -> Dict[str, Any]:
        """Получить текущие метрики как словарь."""
        return self.metrics.to_dict()
    
    def get_summary(self) -> str:
        """Получить сводку метрик в удобочитаемом формате."""
        metrics = self.metrics.to_dict()
        summary = "=" * 60 + "\n"
        summary += "МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ АГЕНТА\n"
        summary += "=" * 60 + "\n"
        summary += f"Всего обработано запросов: {metrics['total_queries']}\n"
        summary += f"Успешных запросов: {metrics['successful_queries']}\n"
        summary += f"Общий процент успешности: {metrics['success_rate']}\n"
        summary += f"\nИспользование инструментов:\n"
        summary += f"  Всего вызовов инструментов: {metrics['total_tool_calls']}\n"
        summary += f"  Успешных вызовов инструментов: {metrics['successful_tool_calls']}\n"
        summary += f"  Процент успешности инструментов: {metrics['tool_success_rate']}\n"
        summary += f"\nПроизводительность:\n"
        summary += f"  Среднее время выполнения: {metrics['average_execution_time']}\n"
        summary += f"  Общее время выполнения: {self.metrics.total_execution_time:.4f}s\n"
        summary += f"\nРаспределение инструментов:\n"
        for tool, count in metrics['tool_usage_distribution'].items():
            success_rate = metrics['tool_success_rates'].get(tool, 'N/A')
        summary += f"\nСложность запросов:\n"
        for complexity, count in metrics['query_complexity'].items():
            summary += f"  • {complexity}: {count} запросов\n"


print("✓ Система метрик и оценки инициализирована")

✓ Система метрик и оценки инициализирована


## Демо: Запуск AI Агента

Давайте инициализируем агента и обработаем некоторые запросы о продуктах маркетплейса.

In [6]:
# Initialize the agent
agent = SimpleAgent(tools=TOOLS, use_llm=False)
metrics_collector = MetricsCollector()

# Test queries
test_queries = [
    "Find the best laptops on the marketplace",
    "Search for smartphones with good prices",
    "Compare the top 3 headphones for me",
    "What headphones do you recommend?",
    "Show me all available products",
]

print("Starting agent execution with test queries...\n")
print("=" * 80)

# Process each query
for i, query in enumerate(test_queries, 1):
    print(f"\nQuery {i}: {query}")
    print("-" * 80)
    
    response = agent.process_query(query)
    print(response)
    
    # Record metrics
    if agent.execution_traces:
        latest_trace = agent.execution_traces[-1]
        metrics_collector.record_trace(latest_trace)
        print(f"\n✓ Executed in {latest_trace.total_execution_time:.4f}s")

print("\n" + "=" * 80)

Starting agent execution with test queries...


Query 1: Find the best laptops on the marketplace
--------------------------------------------------------------------------------
Я помогу вам найти лучшие варианты. Давайте поищем на маркетплейсе.

Найдено 4 продуктов:
  • Dell XPS 13: ₽89,999 (Рейтинг: 4.8/5)
  • MacBook Air M3: ₽119,999 (Рейтинг: 4.9/5)
  • ASUS VivoBook: ₽54,999 (Рейтинг: 4.5/5)
  • HP Pavilion: ₽59,999 (Рейтинг: 4.3/5)


✓ Executed in 0.0000s

Query 2: Search for smartphones with good prices
--------------------------------------------------------------------------------
Я помогу вам найти лучшие варианты. Давайте поищем на маркетплейсе.

Найдено 4 продуктов:
  • iPhone 15 Pro: ₽99,999 (Рейтинг: 4.9/5)
  • Samsung Galaxy S24: ₽79,999 (Рейтинг: 4.7/5)
  • Google Pixel 8: ₽69,999 (Рейтинг: 4.6/5)
  • Xiaomi 14: ₽49,999 (Рейтинг: 4.4/5)


✓ Executed in 0.0000s

Query 3: Compare the top 3 headphones for me
----------------------------------------------------------------

## Метрики производительности агента

Ниже приведены собранные метрики из выполнения агента:

In [7]:
# Display detailed metrics
print(metrics_collector.get_summary())

# Display as JSON for programmatic access
import json
print("\nDetailed Metrics (JSON format):")
print(json.dumps(metrics_collector.get_metrics(), indent=2))

None

Detailed Metrics (JSON format):
{
  "total_queries": 5,
  "successful_queries": 5,
  "success_rate": "100.00%",
  "total_tool_calls": 5,
  "successful_tool_calls": 5,
  "tool_success_rate": "100.00%",
  "average_execution_time": "0.0000s",
  "tool_usage_distribution": {
    "search_marketplace": 4,
    "compare_products": 1
  },
  "tool_success_rates": {
    "search_marketplace": "100.00%",
    "compare_products": "100.00%"
  },
  "query_complexity": {
    "simple": 5,
    "medium": 0,
    "complex": 0
  }
}


## Трассировки выполнения

Подробная разбивка каждого выполнения запроса:

In [8]:
for i, trace in enumerate(agent.execution_traces, 1):
    print(f"\n{'='*80}")
    print(f"Execution Trace #{i}: \"{trace.user_query}\"")
    print(f"{'='*80}")
    
    trace_dict = trace.to_dict()
    print(f"Started: {trace.started_at.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Duration: {trace_dict['duration']:.4f}s")
    print(f"Total Execution Time: {trace_dict['total_execution_time']:.4f}s")
    print(f"Number of Actions: {trace_dict['num_actions']}")
    
    print(f"\nActions:")
    for j, action in enumerate(trace_dict['actions'], 1):
        status = "✓" if action['success'] else "✗"
        print(f"  {j}. {status} {action['tool']} ({action['execution_time']:.4f}s)")
        print(f"     Input: {action['input']}")
    
    print(f"\nResponse:\n{trace.final_response[:500]}..."  if len(trace.final_response) > 500 else f"\nResponse:\n{trace.final_response}")


Execution Trace #1: "Find the best laptops on the marketplace"
Started: 2026-04-22 14:29:51
Duration: 0.0000s
Total Execution Time: 0.0000s
Number of Actions: 1

Actions:
  1. ✓ search_marketplace (0.0000s)
     Input: {'product_category': 'laptop'}

Response:
Я помогу вам найти лучшие варианты. Давайте поищем на маркетплейсе.

Найдено 4 продуктов:
  • Dell XPS 13: ₽89,999 (Рейтинг: 4.8/5)
  • MacBook Air M3: ₽119,999 (Рейтинг: 4.9/5)
  • ASUS VivoBook: ₽54,999 (Рейтинг: 4.5/5)
  • HP Pavilion: ₽59,999 (Рейтинг: 4.3/5)


Execution Trace #2: "Search for smartphones with good prices"
Started: 2026-04-22 14:29:51
Duration: 0.0000s
Total Execution Time: 0.0000s
Number of Actions: 1

Actions:
  1. ✓ search_marketplace (0.0000s)
     Input: {'product_category': 'smartphone'}

Response:
Я помогу вам найти лучшие варианты. Давайте поищем на маркетплейсе.

Найдено 4 продуктов:
  • iPhone 15 Pro: ₽99,999 (Рейтинг: 4.9/5)
  • Samsung Galaxy S24: ₽79,999 (Рейтинг: 4.7/5)
  • Google Pixel 8: ₽69,9

## Расширенные возможности

### 1. Реализация пользовательского инструмента

Вы можете добавить свои собственные инструменты, создав объект `Tool` с:
- **name**: Уникальный идентификатор для инструмента
- **description**: Что делает инструмент (используется LLM)
- **parameters**: JSON схема для параметров
- **function**: Фактическая функция Python для вызова

### 2. Интеграция с реальными API

Для использования реальных API маркетплейса (например, реального API Ozon):
1. Замените имитационные функции реальными вызовами API
2. Добавьте аутентификацию (API ключи, токены)
3. Обработайте ограничение скорости и ответы об ошибках
4. Кэшируйте результаты для производительности

### 3. Объяснение метрик оценки

- **Процент успешности**: % запросов, где все инструменты удались
- **Процент успешности инструментов**: Процент успешности для каждого отдельного инструмента
- **Время выполнения**: Сколько времени заняли вызовы инструментов
- **Сложность запроса**: Классификация на основе количества используемых инструментов
- **Распределение инструментов**: Как часто используется каждый инструмент

### 4. Расширение агента

Класс `SimpleAgent` может быть расширен для:
- Использования реальных LLM (OpenAI, Claude и т.д.)
- Реализации многоходовых разговоров
- Добавления управления памятью/контекстом
- Поддержки потоковых ответов
- Реализации стратегий цепочки инструментов

## Пример использования: Интерактивные запросы

Попробуйте задать агенту пользовательские вопросы:

In [9]:
# Функция для запроса агента и получения метрик
def query_agent_and_analyze(query: str) -> None:
    """Запросить агента и отобразить как ответ, так и метрики."""
    print(f"\n{'='*80}")
    print(f"Запрос: {query}")
    print('='*80)
    
    # Получить ответ
    response = agent.process_query(query)
    print(response)
    
    # Записать и отобразить метрики
    if agent.execution_traces:
        latest_trace = agent.execution_traces[-1]
        metrics_collector.record_trace(latest_trace)
        
        # Отобразить детали трассировки
        trace_dict = latest_trace.to_dict()
        print(f"\nСводка выполнения:")
        print(f"  Длительность: {trace_dict['duration']:.4f}s")
        print(f"  Действия: {trace_dict['num_actions']}")
        print(f"  Успех: {'Да' if all(a['success'] for a in trace_dict['actions']) else 'Нет'}")


# Примеры пользовательских запросов
print("Дополнительные запросы агента:\n")

# Запрос 1
query_agent_and_analyze("Какой самый дешевый смартфон доступен?")

# Запрос 2
query_agent_and_analyze("Найди мне ноутбук с лучшим рейтингом")

# Отобразить обновленные метрики
print(metrics_collector.get_summary())

Дополнительные запросы агента:


Запрос: Какой самый дешевый смартфон доступен?
Я помогу вам найти лучшие варианты. Давайте поищем на маркетплейсе.

Найдено 4 продуктов:
  • Dell XPS 13: ₽89,999 (Рейтинг: 4.8/5)
  • MacBook Air M3: ₽119,999 (Рейтинг: 4.9/5)
  • ASUS VivoBook: ₽54,999 (Рейтинг: 4.5/5)
  • HP Pavilion: ₽59,999 (Рейтинг: 4.3/5)


Сводка выполнения:
  Длительность: 0.0000s
  Действия: 1
  Успех: Да

Запрос: Найди мне ноутбук с лучшим рейтингом
Я помогу вам найти лучшие варианты. Давайте поищем на маркетплейсе.

Найдено 4 продуктов:
  • Dell XPS 13: ₽89,999 (Рейтинг: 4.8/5)
  • MacBook Air M3: ₽119,999 (Рейтинг: 4.9/5)
  • ASUS VivoBook: ₽54,999 (Рейтинг: 4.5/5)
  • HP Pavilion: ₽59,999 (Рейтинг: 4.3/5)


Сводка выполнения:
  Длительность: 0.0000s
  Действия: 1
  Успех: Да
None
